# 第17章 高级安全 - 数字证书与安全协议
# (Digital Certificates & Security Protocols)

## Cambridge A-Level Computer Science (9618)

---

### 本节学习目标

| 知识点 | 英文术语 | 重要程度 |
|--------|----------|----------|
| 数字证书 | Digital Certificates | ★★★★★ |
| 证书颁发机构 | Certificate Authority (CA) | ★★★★★ |
| 数字签名 | Digital Signatures | ★★★★★ |
| TLS/SSL协议 | TLS/SSL Protocol | ★★★★★ |
| 哈希函数 | Hash Functions | ★★★★ |
| 量子密码学 | Quantum Cryptography | ★★★ |

---
## 1. 哈希函数 (Hash Functions)

在讲数字签名和证书之前，我们必须先理解**哈希函数**，因为它是后面所有内容的基础。

### 什么是哈希函数？

哈希函数是一种将**任意长度的输入**转换为**固定长度输出**的单向函数。

### 哈希函数的关键特性

1. **确定性 (Deterministic)**：相同输入永远产生相同输出
2. **单向性 (One-way)**：从哈希值无法反推出原始输入
3. **雪崩效应 (Avalanche Effect)**：输入的微小变化导致输出的巨大变化
4. **抗碰撞性 (Collision Resistance)**：几乎不可能找到两个不同的输入产生相同的哈希值

### 为什么哈希函数如此重要？

因为它能让我们**验证数据的完整性**而**不暴露数据本身**。
- 你可以公开一个文件的哈希值，别人下载后计算哈希值对比，就知道文件有没有被篡改
- 数据库中存储密码的哈希值而非明文密码
- 数字签名中，我们对**数据的哈希值**进行签名，而非数据本身

In [ ]:
import hashlib

print('='*60)
print('SHA-256 哈希函数演示 (Hash Function Demo)')
print('='*60)
print()

# 基本哈希演示
messages = [
    'Hello World',
    'Hello world',   # 只改了一个大小写!
    'Hello World!',  # 只加了一个感叹号!
    'A' * 1000,      # 长输入
    'B',             # 短输入
]

print('--- SHA-256 哈希值 ---')
print(f'{"输入":.<35} SHA-256 哈希值')
print('-' * 100)
for msg in messages:
    display = msg if len(msg) <= 30 else msg[:27] + '...'
    hash_val = hashlib.sha256(msg.encode()).hexdigest()
    print(f'{display:.<35} {hash_val}')

print()
print('关键观察:')
print('1. 所有输出都是64个十六进制字符(256位) -- 固定长度!')
print('2. "Hello World" vs "Hello world" -- 只改了一个字母, 哈希完全不同!')
print('   这就是"雪崩效应" (Avalanche Effect)')


In [ ]:
import matplotlib.pyplot as plt
import hashlib
import numpy as np

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 可视化雪崩效应
def hash_to_bits(text):
    """将文本的SHA-256哈希转换为位数组"""
    h = hashlib.sha256(text.encode()).digest()
    bits = []
    for byte in h:
        for i in range(7, -1, -1):
            bits.append((byte >> i) & 1)
    return np.array(bits)

# 比较两个相似输入的哈希
text1 = 'Hello World'
text2 = 'Hello world'  # 只改了W->w

bits1 = hash_to_bits(text1)
bits2 = hash_to_bits(text2)
diff = bits1 != bits2  # 不同的位

fig, axes = plt.subplots(3, 1, figsize=(16, 6))

# 重新整形为16x16便于显示
axes[0].imshow(bits1.reshape(16, 16), cmap='Blues', aspect='auto')
axes[0].set_title(f'SHA-256("{text1}") -- 哈希值的位图', fontsize=12, fontweight='bold')
axes[0].set_ylabel('位')

axes[1].imshow(bits2.reshape(16, 16), cmap='Greens', aspect='auto')
axes[1].set_title(f'SHA-256("{text2}") -- 仅改了一个字母!', fontsize=12, fontweight='bold')
axes[1].set_ylabel('位')

axes[2].imshow(diff.reshape(16, 16), cmap='Reds', aspect='auto')
axes[2].set_title(f'差异位图 (红色=不同) -- {diff.sum()}/256位不同 ({diff.sum()/256*100:.1f}%)', 
                  fontsize=12, fontweight='bold')
axes[2].set_ylabel('位')

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

print(f'输入只改了1个字符, 但哈希值有 {diff.sum()}/256 ({diff.sum()/256*100:.1f}%) 的位发生了变化!')


---
## 2. 数字签名 (Digital Signatures)

### 为什么需要数字签名？

假设Alice收到一封声称来自银行的邮件，要求她转账。她怎么确认：
1. **这真的是银行发的**（身份认证，Authentication）
2. **内容没有被篡改**（完整性，Integrity）
3. **银行以后不能否认发过这封邮件**（不可否认性，Non-repudiation）

这三个问题，**数字签名**都能解决！

### 数字签名的原理

数字签名利用了非对称加密的**反向操作**：
- 正常加密：公钥加密 -> 私钥解密
- 数字签名：**私钥签名 -> 公钥验证**

### 签名流程
1. 发送方对消息计算**哈希值**
2. 用自己的**私钥**加密哈希值 -> 这就是"数字签名"
3. 将消息 + 签名一起发送

### 验证流程
1. 接收方用发送方的**公钥**解密签名 -> 得到发送方计算的哈希值
2. 接收方自己对收到的消息计算哈希值
3. 比较两个哈希值：如果相同，则消息未被篡改且确实来自发送方

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 1, figsize=(15, 12))

# === 签名过程 ===
ax = axes[0]
ax.set_xlim(0, 15)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title('数字签名过程 (Digital Signature - Signing)', fontsize=14, fontweight='bold', color='#1565C0')

# 发送方
circle = plt.Circle((1.2, 3), 0.8, color='#1565C0', ec='black', lw=2)
ax.add_patch(circle)
ax.text(1.2, 3, 'Alice\n(发送方)', ha='center', va='center', fontsize=9, color='white', fontweight='bold')

# 消息
r1 = mpatches.FancyBboxPatch((2.5, 3.8), 2.0, 1.2, boxstyle='round,pad=0.15',
                              facecolor='#4CAF50', edgecolor='black', lw=1.5)
ax.add_patch(r1)
ax.text(3.5, 4.4, 'Message\n(消息)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# 哈希
ax.annotate('', xy=(5.0, 4.4), xytext=(4.6, 4.4),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
r2 = mpatches.FancyBboxPatch((5.0, 3.8), 2.2, 1.2, boxstyle='round,pad=0.15',
                              facecolor='#FF9800', edgecolor='black', lw=1.5)
ax.add_patch(r2)
ax.text(6.1, 4.4, 'Hash\nFunction', ha='center', va='center', fontsize=10, fontweight='bold')

# 哈希值
ax.annotate('', xy=(7.8, 4.4), xytext=(7.3, 4.4),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
r3 = mpatches.FancyBboxPatch((7.8, 3.8), 1.8, 1.2, boxstyle='round,pad=0.15',
                              facecolor='#FF5722', edgecolor='black', lw=1.5)
ax.add_patch(r3)
ax.text(8.7, 4.4, 'Hash\n(摘要)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# 私钥加密
ax.annotate('', xy=(10.2, 4.4), xytext=(9.7, 4.4),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
r4 = mpatches.FancyBboxPatch((10.2, 3.8), 2.2, 1.2, boxstyle='round,pad=0.15',
                              facecolor='#E91E63', edgecolor='black', lw=1.5)
ax.add_patch(r4)
ax.text(11.3, 4.4, 'Encrypt\nw/ Private Key\n(私钥加密)', ha='center', va='center', fontsize=8, color='white', fontweight='bold')

# 签名输出
ax.annotate('', xy=(13.0, 4.4), xytext=(12.5, 4.4),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
r5 = mpatches.FancyBboxPatch((13.0, 3.8), 1.5, 1.2, boxstyle='round,pad=0.15',
                              facecolor='#9C27B0', edgecolor='black', lw=1.5)
ax.add_patch(r5)
ax.text(13.75, 4.4, 'Signature\n(签名)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# 发送内容
r6 = mpatches.FancyBboxPatch((4.0, 0.8), 7.0, 2.0, boxstyle='round,pad=0.2',
                              facecolor='#E8EAF6', edgecolor='#3F51B5', lw=2, ls='--')
ax.add_patch(r6)
ax.text(7.5, 2.1, 'Sent Together (一起发送)', ha='center', va='center', fontsize=11, fontweight='bold', color='#3F51B5')
ax.text(6.0, 1.4, 'Message + Signature', ha='center', va='center', fontsize=12, fontweight='bold')

# === 验证过程 ===
ax = axes[1]
ax.set_xlim(0, 15)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title('数字签名验证 (Digital Signature - Verification)', fontsize=14, fontweight='bold', color='#2E7D32')

# 接收方
circle2 = plt.Circle((1.2, 3), 0.8, color='#E91E63', ec='black', lw=2)
ax.add_patch(circle2)
ax.text(1.2, 3, 'Bob\n(接收方)', ha='center', va='center', fontsize=9, color='white', fontweight='bold')

# 路径1: 消息 -> 哈希
r1 = mpatches.FancyBboxPatch((2.8, 4.0), 1.8, 1.0, boxstyle='round,pad=0.15',
                              facecolor='#4CAF50', edgecolor='black', lw=1.5)
ax.add_patch(r1)
ax.text(3.7, 4.5, 'Message', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

ax.annotate('', xy=(5.2, 4.5), xytext=(4.7, 4.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
r2 = mpatches.FancyBboxPatch((5.2, 4.0), 2.0, 1.0, boxstyle='round,pad=0.15',
                              facecolor='#FF9800', edgecolor='black', lw=1.5)
ax.add_patch(r2)
ax.text(6.2, 4.5, 'Hash Fn', ha='center', va='center', fontsize=10, fontweight='bold')

ax.annotate('', xy=(7.8, 4.5), xytext=(7.3, 4.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
r3 = mpatches.FancyBboxPatch((7.8, 4.0), 1.8, 1.0, boxstyle='round,pad=0.15',
                              facecolor='#FF5722', edgecolor='black', lw=1.5)
ax.add_patch(r3)
ax.text(8.7, 4.5, 'Hash A', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# 路径2: 签名 -> 公钥解密
r4 = mpatches.FancyBboxPatch((2.8, 1.0), 1.8, 1.0, boxstyle='round,pad=0.15',
                              facecolor='#9C27B0', edgecolor='black', lw=1.5)
ax.add_patch(r4)
ax.text(3.7, 1.5, 'Signature', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

ax.annotate('', xy=(5.2, 1.5), xytext=(4.7, 1.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
r5 = mpatches.FancyBboxPatch((5.2, 1.0), 2.0, 1.0, boxstyle='round,pad=0.15',
                              facecolor='#2196F3', edgecolor='black', lw=1.5)
ax.add_patch(r5)
ax.text(6.2, 1.5, 'Decrypt\nw/ Public Key', ha='center', va='center', fontsize=9, color='white', fontweight='bold')

ax.annotate('', xy=(7.8, 1.5), xytext=(7.3, 1.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
r6 = mpatches.FancyBboxPatch((7.8, 1.0), 1.8, 1.0, boxstyle='round,pad=0.15',
                              facecolor='#FF5722', edgecolor='black', lw=1.5)
ax.add_patch(r6)
ax.text(8.7, 1.5, 'Hash B', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# 比较
ax.annotate('', xy=(10.5, 3.0), xytext=(9.7, 4.3),
            arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=2.5))
ax.annotate('', xy=(10.5, 3.0), xytext=(9.7, 1.7),
            arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=2.5))

r7 = mpatches.FancyBboxPatch((10.3, 2.2), 3.5, 1.6, boxstyle='round,pad=0.2',
                              facecolor='#4CAF50', edgecolor='black', lw=2)
ax.add_patch(r7)
ax.text(12.05, 3.0, 'Hash A == Hash B ?\n\nYes: Valid (有效)\nNo: Invalid (无效)', 
        ha='center', va='center', fontsize=10, color='white', fontweight='bold')

plt.tight_layout()


In [ ]:
import hashlib
import math

print('='*60)
print('简单数字签名模拟 (Digital Signature Simulation)')
print('='*60)
print()

# 使用简化RSA实现数字签名
def simple_rsa_keygen(p, q):
    n = p * q
    phi = (p - 1) * (q - 1)
    e = 17
    while math.gcd(e, phi) != 1:
        e += 2
    d = pow(e, -1, phi)
    return (e, n), (d, n)

# 生成密钥对
public_key, private_key = simple_rsa_keygen(61, 53)
e, n = public_key
d, _ = private_key
print(f'Alice的公钥 (Public Key):  (e={e}, n={n})')
print(f'Alice的私钥 (Private Key): (d={d}, n={n})')
print()

# 签名过程
message = 'Transfer 1000 yuan to Bob'
print(f'原始消息: "{message}"')

# Step 1: 计算哈希
hash_val = hashlib.sha256(message.encode()).hexdigest()
# 取哈希的前几位转为数字（简化）
hash_num = int(hash_val[:4], 16) % n
print(f'消息哈希 (简化): {hash_num}')

# Step 2: 用私钥加密哈希 = 签名
signature = pow(hash_num, d, n)
print(f'数字签名: {signature}')
print()

# 验证过程
print('--- Bob验证签名 ---')

# Step 1: 用Alice的公钥解密签名
decrypted_hash = pow(signature, e, n)
print(f'公钥解密签名得到: {decrypted_hash}')

# Step 2: 自己计算消息哈希
received_hash = int(hashlib.sha256(message.encode()).hexdigest()[:4], 16) % n
print(f'自己计算的哈希:   {received_hash}')

# Step 3: 比较
if decrypted_hash == received_hash:
    print('\nResult: VALID! 签名有效! 消息确实来自Alice且未被篡改!')
else:
    print('\nResult: INVALID! 签名无效! 消息可能被篡改!')

# 演示篡改检测
print()
print('--- 如果消息被篡改? ---')
tampered_msg = 'Transfer 9999 yuan to Eve'
print(f'篡改后消息: "{tampered_msg}"')
tampered_hash = int(hashlib.sha256(tampered_msg.encode()).hexdigest()[:4], 16) % n
print(f'篡改消息的哈希: {tampered_hash}')
print(f'签名解密得到:   {decrypted_hash}')
if tampered_hash != decrypted_hash:


---
## 3. 数字证书 (Digital Certificates)

### 一个关键问题

数字签名能验证"消息确实来自公钥的持有者"。但问题是：**你怎么确定这个公钥确实属于Alice，而不是Eve冒充的？**

这就是**数字证书**要解决的问题。

### 什么是数字证书？

数字证书就像一个"数字身份证"，由受信任的**证书颁发机构 (Certificate Authority, CA)** 签发。

### 证书包含的信息

| 字段 | 英文 | 说明 |
|------|------|------|
| 持有者信息 | Subject | 证书持有者的名称、组织等 |
| 持有者公钥 | Public Key | 持有者的公钥 |
| 颁发者 | Issuer | 颁发此证书的CA |
| 有效期 | Validity Period | 证书的有效起止日期 |
| 序列号 | Serial Number | 唯一标识符 |
| CA的数字签名 | Digital Signature | CA用自己的私钥对以上信息的签名 |

### 证书链 (Certificate Chain)

为什么你的浏览器信任某个CA？因为操作系统和浏览器预装了一些**根证书 (Root Certificates)**。

信任链：**根CA -> 中间CA -> 网站证书**

这就像：国家政府 -> 省政府 -> 市政府发的身份证，你信任国家政府，所以也信任它授权的下级机构。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('证书链 (Certificate Chain of Trust)', fontsize=16, fontweight='bold')

# 根CA
r1 = mpatches.FancyBboxPatch((4.5, 8.0), 5.0, 1.5, boxstyle='round,pad=0.3',
                              facecolor='#D32F2F', edgecolor='black', lw=2)
ax.add_patch(r1)
ax.text(7.0, 8.75, 'Root CA (根证书颁发机构)', ha='center', va='center',
        fontsize=12, color='white', fontweight='bold')
ax.text(7.0, 8.3, 'Pre-installed in OS/Browser', ha='center', va='center',
        fontsize=9, color='#FFCDD2')

# 箭头: 根CA -> 中间CA
ax.annotate('Signs (签发)', xy=(5.5, 6.8), xytext=(5.5, 7.9),
            arrowprops=dict(arrowstyle='->', color='black', lw=2),
            fontsize=10, ha='center', fontweight='bold')
ax.annotate('Signs (签发)', xy=(8.5, 6.8), xytext=(8.5, 7.9),
            arrowprops=dict(arrowstyle='->', color='black', lw=2),
            fontsize=10, ha='center', fontweight='bold')

# 中间CA
r2 = mpatches.FancyBboxPatch((2.5, 5.3), 4.0, 1.5, boxstyle='round,pad=0.2',
                              facecolor='#F57C00', edgecolor='black', lw=2)
ax.add_patch(r2)
ax.text(4.5, 6.05, 'Intermediate CA', ha='center', va='center',
        fontsize=11, color='white', fontweight='bold')
ax.text(4.5, 5.65, '(中间CA - A)', ha='center', va='center', fontsize=9, color='#FFE0B2')

r3 = mpatches.FancyBboxPatch((7.5, 5.3), 4.0, 1.5, boxstyle='round,pad=0.2',
                              facecolor='#F57C00', edgecolor='black', lw=2)
ax.add_patch(r3)
ax.text(9.5, 6.05, 'Intermediate CA', ha='center', va='center',
        fontsize=11, color='white', fontweight='bold')
ax.text(9.5, 5.65, '(中间CA - B)', ha='center', va='center', fontsize=9, color='#FFE0B2')

# 箭头: 中间CA -> 网站证书
ax.annotate('Signs', xy=(3.5, 3.8), xytext=(3.5, 5.2),
            arrowprops=dict(arrowstyle='->', color='black', lw=2),
            fontsize=10, ha='center', fontweight='bold')
ax.annotate('Signs', xy=(5.5, 3.8), xytext=(5.5, 5.2),
            arrowprops=dict(arrowstyle='->', color='black', lw=2),
            fontsize=10, ha='center', fontweight='bold')
ax.annotate('Signs', xy=(9.5, 3.8), xytext=(9.5, 5.2),
            arrowprops=dict(arrowstyle='->', color='black', lw=2),
            fontsize=10, ha='center', fontweight='bold')

# 网站证书
sites = [
    (1.5, 'google.com', '#4CAF50'),
    (4.5, 'baidu.com', '#2196F3'),
    (8.0, 'amazon.com', '#9C27B0'),
]
for x, name, color in sites:
    r = mpatches.FancyBboxPatch((x, 2.3), 3.0, 1.5, boxstyle='round,pad=0.2',
                                facecolor=color, edgecolor='black', lw=2)
    ax.add_patch(r)
    ax.text(x+1.5, 3.05, name, ha='center', va='center',
            fontsize=11, color='white', fontweight='bold')
    ax.text(x+1.5, 2.65, 'Site Certificate', ha='center', va='center',
            fontsize=9, color='white')

# 注释
ax.text(7.0, 1.0, 'Browser验证: 网站证书 -> 中间CA签名有效? -> 中间CA证书 -> 根CA签名有效? -> 根CA已预装 -> 信任!',
        ha='center', va='center', fontsize=10, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#E8F5E9', edgecolor='#4CAF50', lw=2))

plt.tight_layout()


In [ ]:
# 模拟数字证书的结构
import hashlib
import json
from datetime import datetime

print('='*60)
print('数字证书结构模拟 (Digital Certificate Structure)')
print('='*60)
print()

certificate = {
    'version': 'X.509 v3',
    'serial_number': 'A1:B2:C3:D4:E5:F6',
    'issuer': {
        'common_name': 'TrustCA Root Authority',
        'organization': 'TrustCA Inc.',
        'country': 'US'
    },
    'subject': {
        'common_name': '*.example.com',
        'organization': 'Example Corp.',
        'country': 'CN'
    },
    'validity': {
        'not_before': '2025-01-01',
        'not_after': '2026-12-31'
    },
    'public_key': {
        'algorithm': 'RSA',
        'key_size': 2048,
        'modulus': 'A1B2C3D4...（实际中是很长的数字）',
        'exponent': 65537
    },
    'extensions': {
        'key_usage': ['digitalSignature', 'keyEncipherment'],
        'subject_alt_names': ['example.com', '*.example.com']
    },
    'signature_algorithm': 'SHA256withRSA'
}

# 漂亮地打印证书
print(json.dumps(certificate, indent=2, ensure_ascii=False))
print()

# 模拟CA签名
cert_data = json.dumps(certificate, sort_keys=True)
cert_hash = hashlib.sha256(cert_data.encode()).hexdigest()
print(f'证书数据的SHA-256哈希:')
print(f'  {cert_hash}')
print(f'CA会用自己的私钥对这个哈希值进行加密, 生成签名')


---
## 4. TLS/SSL协议 (TLS/SSL Protocol)

### 你每次打开https网站时，背后发生了什么？

当你在浏览器中输入 `https://www.google.com` 时，在你看到页面之前，浏览器和服务器之间进行了一系列复杂的"握手 (Handshake)"过程。这个过程使用了我们前面学的**所有技术**：

- 非对称加密 (用于密钥交换)
- 对称加密 (用于数据传输)
- 数字证书 (用于验证服务器身份)
- 哈希函数 (用于数据完整性)

### TLS握手的简化步骤

1. **Client Hello**: 浏览器发送支持的加密算法列表和随机数
2. **Server Hello**: 服务器选择加密算法，发送数字证书和随机数
3. **Certificate Verify**: 浏览器验证证书（通过证书链）
4. **Key Exchange**: 双方协商一个对称密钥（会话密钥）
5. **Finished**: 双方用会话密钥进行对称加密通信

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(14, 12))
ax.set_xlim(0, 14)
ax.set_ylim(0, 14)
ax.axis('off')
ax.set_title('TLS/SSL 握手过程 (TLS Handshake)', fontsize=16, fontweight='bold')

# 浏览器和服务器
browser_x, server_x = 2.5, 11.5

# 浏览器图标
r_browser = mpatches.FancyBboxPatch((1.5, 12.5), 2.0, 1.2, boxstyle='round,pad=0.2',
                                     facecolor='#2196F3', edgecolor='black', lw=2)
ax.add_patch(r_browser)
ax.text(browser_x, 13.1, 'Browser\n(浏览器)', ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# 服务器图标
r_server = mpatches.FancyBboxPatch((10.5, 12.5), 2.0, 1.2, boxstyle='round,pad=0.2',
                                    facecolor='#4CAF50', edgecolor='black', lw=2)
ax.add_patch(r_server)
ax.text(server_x, 13.1, 'Server\n(服务器)', ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# 垂直时间线
ax.plot([browser_x, browser_x], [12.4, 0.5], color='#2196F3', lw=2, ls='--', alpha=0.5)
ax.plot([server_x, server_x], [12.4, 0.5], color='#4CAF50', lw=2, ls='--', alpha=0.5)

# 步骤
steps = [
    (11.5, browser_x, server_x, '#FF5722', '1. Client Hello',
     'Supported ciphers + Random number\n(支持的加密算法 + 随机数)'),
    (9.5, server_x, browser_x, '#9C27B0', '2. Server Hello + Certificate',
     'Chosen cipher + Certificate + Random number\n(选定算法 + 证书 + 随机数)'),
    (7.5, browser_x, browser_x, '#FFC107', '3. Verify Certificate',
     'Check certificate chain -> CA trusted?\n(验证证书链 -> CA可信?)'),
    (5.5, browser_x, server_x, '#E91E63', '4. Key Exchange',
     'Pre-master secret encrypted w/ server public key\n(用服务器公钥加密预主密钥)'),
    (3.5, browser_x, server_x, '#00BCD4', '5. Both derive Session Key',
     'Session Key = f(pre-master, random1, random2)\n(双方计算出相同的会话密钥)'),
    (1.5, browser_x, server_x, '#4CAF50', '6. Encrypted Communication',
     'All data encrypted with Symmetric Session Key\n(所有数据用对称会话密钥加密)'),
]

for y, x1, x2, color, title, desc in steps:
    # 箭头
    if x1 != x2:
        ax.annotate('', xy=(x2, y), xytext=(x1, y),
                    arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    
    # 标签
    mid_x = (x1 + x2) / 2 if x1 != x2 else x1 + 2.5
    ax.text(mid_x, y + 0.35, title, ha='center', va='center',
            fontsize=10, fontweight='bold', color=color)
    ax.text(mid_x, y - 0.35, desc, ha='center', va='center',
            fontsize=8, color='#424242')

# 特殊标注: 步骤3是浏览器内部操作
ax.add_patch(mpatches.FancyBboxPatch((1.2, 7.0), 4.5, 1.0, boxstyle='round,pad=0.15',
                                      facecolor='#FFF9C4', edgecolor='#FFC107', lw=1.5, ls='--'))

# 分隔线: 非对称 vs 对称
ax.axhline(y=4.5, xmin=0.05, xmax=0.95, color='red', lw=1.5, ls=':')
ax.text(7.0, 4.5, '--- 以上: 非对称加密 (慢) | 以下: 对称加密 (快) ---',
        ha='center', va='center', fontsize=10, color='red', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='red'))

plt.tight_layout()


In [ ]:
# 模拟TLS握手过程
import hashlib
import os
import math

print('='*60)
print('TLS握手过程模拟 (Simulated TLS Handshake)')
print('='*60)
print()

# 模拟RSA密钥对
def rsa_keygen(p, q):
    n = p * q
    phi = (p-1)*(q-1)
    e = 17
    while math.gcd(e, phi) != 1:
        e += 2
    d = pow(e, -1, phi)
    return (e, n), (d, n)

server_pub, server_priv = rsa_keygen(61, 53)

# Step 1: Client Hello
print('Step 1: Client Hello')
client_random = int.from_bytes(os.urandom(4), 'big') % 1000
supported_ciphers = ['AES-256-GCM', 'AES-128-CBC', 'ChaCha20']
print(f'  Client Random: {client_random}')
print(f'  Supported Ciphers: {supported_ciphers}')
print()

# Step 2: Server Hello
print('Step 2: Server Hello')
server_random = int.from_bytes(os.urandom(4), 'big') % 1000
chosen_cipher = 'AES-256-GCM'
print(f'  Server Random: {server_random}')
print(f'  Chosen Cipher: {chosen_cipher}')
print(f'  Server Certificate: CN=example.com, PubKey={server_pub}')
print()

# Step 3: Verify Certificate
print('Step 3: Browser Verifies Certificate')
print('  Checking certificate chain...')
print('  Root CA signature: VALID')
print('  Certificate not expired: VALID')
print('  Domain matches: VALID')
print('  Certificate: TRUSTED')
print()

# Step 4: Key Exchange
print('Step 4: Key Exchange')
pre_master_secret = int.from_bytes(os.urandom(4), 'big') % (server_pub[1] - 1)
print(f'  Pre-master secret: {pre_master_secret}')
e, n = server_pub
encrypted_pms = pow(pre_master_secret, e, n)
print(f'  Encrypted with server public key: {encrypted_pms}')
print(f'  (Only server can decrypt with private key)')
print()

# Server decrypts
d, n = server_priv
decrypted_pms = pow(encrypted_pms, d, n)
print(f'  Server decrypts: {decrypted_pms}')
print(f'  Match: {pre_master_secret == decrypted_pms}')
print()

# Step 5: Derive session key
print('Step 5: Derive Session Key')
key_material = f'{pre_master_secret}{client_random}{server_random}'
session_key = hashlib.sha256(key_material.encode()).hexdigest()[:32]
print(f'  Session Key = SHA256(pre_master + client_random + server_random)')
print(f'  Session Key: {session_key}')
print()

# Step 6: Encrypted communication
print('Step 6: Encrypted Communication (AES with Session Key)')
print(f'  All subsequent data encrypted with session key: {session_key[:16]}...')
print('  -> GET / HTTP/1.1  [ENCRYPTED]')
print('  <- HTTP/1.1 200 OK [ENCRYPTED]')
print()


---
## 5. 量子密码学简介 (Quantum Cryptography)

### 为什么量子计算可能打破现有加密？

RSA的安全性基于"大质数分解很难"。但这个"难"是对**经典计算机**而言的。

**量子计算机 (Quantum Computer)** 利用量子力学原理，可以使用 **Shor算法 (Shor's Algorithm)** 在多项式时间内完成大数分解！

这意味着：
- RSA加密 -> 不再安全
- 基于离散对数的加密（如Diffie-Hellman、ECC）-> 不再安全
- 对称加密（如AES）-> 安全性减半（AES-256仍然足够安全）

### 量子密钥分发 (Quantum Key Distribution, QKD)

量子密码学利用量子力学的**两个关键原理**：

1. **测量改变状态 (Measurement Disturbs State)**：观测一个量子比特会不可避免地改变它的状态
2. **不可克隆定理 (No-Cloning Theorem)**：无法完美复制一个未知的量子态

这意味着：如果有人试图窃听量子密钥传输，接收方**一定能检测到**，因为窃听会改变量子态！

### BB84协议 (简化版)

1. Alice随机选择基(basis)发送量子比特
2. Bob随机选择基来测量
3. 他们公开比较用了哪些基（但不公开比特值）
4. 只保留基匹配的比特作为密钥
5. 如果有窃听者，错误率会明显升高 -> 检测到窃听！

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 经典计算机 vs 量子计算机破解RSA所需时间
fig, ax = plt.subplots(figsize=(12, 7))

key_sizes = [512, 1024, 2048, 4096]
# 经典计算机估计时间(年) - 指数增长
classical_time = [0.001, 100, 1e15, 1e30]
# 量子计算机估计时间(年) - 多项式增长
quantum_time = [0.000001, 0.001, 0.01, 0.1]

x = np.arange(len(key_sizes))
width = 0.35

bars1 = ax.bar(x - width/2, classical_time, width, label='Classical Computer (经典计算机)',
               color='#2196F3', edgecolor='black', alpha=0.8)
bars2 = ax.bar(x + width/2, quantum_time, width, label='Quantum Computer (量子计算机)',
               color='#F44336', edgecolor='black', alpha=0.8)

ax.set_yscale('log')
ax.set_xlabel('RSA Key Size (bits)', fontsize=12)
ax.set_ylabel('Estimated Time (years, log scale)', fontsize=12)
ax.set_title('RSA破解时间: 经典计算机 vs 量子计算机', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in key_sizes])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# 宇宙年龄参考线
ax.axhline(y=1.38e10, color='orange', ls='--', lw=1.5)
ax.text(0.5, 2e10, 'Age of Universe (宇宙年龄: 138亿年)', fontsize=9, color='orange')

plt.tight_layout()
plt.show()

print('RSA-2048用经典计算机需要远超宇宙年龄的时间来破解')
print('但量子计算机(理论上)可以在很短时间内完成!')


In [ ]:
import random

print('='*60)
print('BB84量子密钥分发协议模拟')
print('(BB84 Quantum Key Distribution Protocol Simulation)')
print('='*60)
print()

random.seed(42)
n_bits = 20  # 发送20个量子比特

# Alice随机选择比特和基
alice_bits  = [random.randint(0, 1) for _ in range(n_bits)]
alice_bases = [random.choice(['+', 'x']) for _ in range(n_bits)]

# Bob随机选择基
bob_bases = [random.choice(['+', 'x']) for _ in range(n_bits)]

# Bob测量: 基相同则测量正确, 基不同则随机结果
bob_bits = []
for i in range(n_bits):
    if alice_bases[i] == bob_bases[i]:
        bob_bits.append(alice_bits[i])  # 基匹配, 测量正确
    else:
        bob_bits.append(random.randint(0, 1))  # 基不匹配, 随机

# 公开比较基
print('Alice bits:  ', alice_bits)
print('Alice bases: ', alice_bases)
print('Bob bases:   ', bob_bases)
print('Bob bits:    ', bob_bits)
print()

# 筛选基匹配的比特
shared_key = []
print('基匹配分析:')
for i in range(n_bits):
    match = 'MATCH' if alice_bases[i] == bob_bases[i] else 'DIFF'
    if match == 'MATCH':
        shared_key.append(alice_bits[i])
    mark = '<- KEY' if match == 'MATCH' else ''
    print(f'  Bit {i:2d}: Alice({alice_bits[i]}, {alice_bases[i]}) '
          f'Bob({bob_bits[i]}, {bob_bases[i]}) -> {match} {mark}')

print(f'\nShared Key (共享密钥): {shared_key}')
print(f'密钥长度: {len(shared_key)} bits (从 {n_bits} bits 中提取)')
print()

# 模拟窃听检测
print('--- 窃听检测 ---')
print('如果有窃听者Eve, 她的测量会干扰量子态,')
print('导致Bob即使用正确的基也会得到错误结果。')
print('Alice和Bob可以公开比较一小部分密钥来检查错误率。')
print('正常情况: 错误率 = 0%')


---
## 6. 知识总结

### 核心考点 (Key Exam Points)

1. **哈希函数**是单向的，具有确定性、雪崩效应和抗碰撞性
2. **数字签名** = 用私钥加密消息的哈希值，提供身份认证、完整性和不可否认性
3. **数字证书**由CA签发，绑定公钥与身份，形成信任链
4. **TLS/SSL**是HTTPS的安全基础，结合了非对称加密（密钥交换）和对称加密（数据传输）
5. **量子计算**可能用Shor算法破解RSA等基于因数分解的加密
6. **量子密钥分发(QKD)**利用量子力学原理，任何窃听都会被检测到

### 技术之间的关系

```
哈希函数 -> 数字签名 -> 数字证书 -> TLS/SSL -> HTTPS
   |            |           |          |
完整性      身份认证     信任链    安全通信
```

---
## 7. 练习题 (Exercises)

### 练习1：哈希函数
解释为什么数据库中应该存储密码的哈希值而不是明文密码。如果两个用户使用相同的密码，他们的哈希值会相同吗？这有什么安全隐患？如何解决？(提示：Salt)

### 练习2：数字签名
Alice给Bob发送了一条带数字签名的消息。中间人Eve截获了消息和签名。
- (a) Eve能验证签名吗？（她有Alice的公钥）
- (b) Eve能修改消息而不被发现吗？为什么？
- (c) Eve能伪造Alice的签名吗？为什么？

### 练习3：TLS握手
按顺序排列TLS握手的步骤，并解释每个步骤使用了什么加密技术（对称/非对称/哈希）。

### 练习4：量子威胁
解释为什么量子计算机对RSA是威胁，但对AES-256的威胁相对较小。

### 练习5：编程挑战
编写一个程序，实现简单的"文件完整性检查器"：
1. 计算给定字符串的SHA-256哈希
2. 修改字符串中的一个字符
3. 重新计算哈希并比较
4. 报告是否检测到修改

In [ ]:
# 练习1: 密码哈希与Salt
import hashlib
import os

# 不安全: 直接哈希
password = 'mypassword123'
simple_hash = hashlib.sha256(password.encode()).hexdigest()
print(f'简单哈希: {simple_hash}')
print('问题: 相同密码产生相同哈希, 容易被彩虹表攻击!')
print()

# 安全: 加Salt后哈希
# salt = os.urandom(16).hex()  # 随机生成salt
# salted_hash = hashlib.sha256((salt + password).encode()).hexdigest()
# print(f'Salt: {salt}')
# print(f'加Salt的哈希: {salted_hash}')

# 请完成以上代码并思考:


In [ ]:
# 练习5: 文件完整性检查器
import hashlib

def integrity_checker(original, modified):
    """
    检查两个字符串的完整性
    返回: (original_hash, modified_hash, is_same)
    """
    # 请在这里实现你的代码
    pass

# 测试
# result = integrity_checker('Hello World', 'Hello world')
